![Cabecera](../../assets/cabecera_rag.png)

# Workout 3 — Evaluar y ajustar el Retrieval

## Objetivos

Un retriever puede funcionar con **una** pregunta de demo y fallar con otras. Aquí aprendes a **evaluarlo** antes de conectar un LLM (Sprint 10).

Evaluar retrieval significa responder:

- ¿Los chunks recuperados **contienen** la información necesaria?
- ¿La **fuente** (`FAQ`, `CSV`, `PDF`) es la esperada?
- ¿Cambiar **K** (cuántos chunks traemos) mejora el resultado?

> **No evaluamos respuestas redactadas** — eso sería mezclar retrieval con generación. Solo miramos el **contexto recuperado**.

**Prerrequisitos:** Workouts 1 y 2 ejecutados (`../output/chroma_db/` existe).

## Setup — Instalar librerías

Mismas dependencias que en el Workout 2: Chroma para leer el índice y Gemini para embeddear cada pregunta de prueba.

In [ ]:
%pip install -q chromadb google-genai

## Preparar conexión, preguntas de prueba y funciones auxiliares

En esta celda hacemos tres cosas:

1. **Conectar** al índice Chroma del Workout 1.
2. **Cargar** preguntas fijas desde `../queries/preguntas_eval.json` (nuestro mini-benchmark).
3. **Definir funciones** que usaremos en todo el notebook:
   - `embeddear_consulta()` — pregunta → vector.
   - `recuperar()` — vector → top-K chunks con distancia y metadata.
   - `formatear_contexto()` — chunks → texto legible.

Variables, conexiones y carga de datos:

In [ ]:
import json
import os
from getpass import getpass
from pathlib import Path

import chromadb
from dotenv import load_dotenv
from google import genai
from google.genai import types

WORKOUT_DIR = Path("..").resolve()
CHROMA_DIR = WORKOUT_DIR / "output" / "chroma_db"
QUERIES_JSON = WORKOUT_DIR / "queries" / "preguntas_eval.json"
COLLECTION_NAME = "agenda_cultural_madrid"
EMBEDDING_MODEL = "gemini-embedding-2"

if not os.getenv("GEMINI_API_KEY"):
    os.environ["GEMINI_API_KEY"] = getpass("GEMINI_API_KEY: ")

assert CHROMA_DIR.exists(), "Ejecuta el Workout 1 primero."

client = genai.Client()
chroma = chromadb.PersistentClient(path=str(CHROMA_DIR))
collection = chroma.get_collection(COLLECTION_NAME)

preguntas = json.loads(QUERIES_JSON.read_text(encoding="utf-8"))["preguntas"]
print(f"Índice: {collection.count()} docs | Preguntas eval: {len(preguntas)}")

Índice: 20 docs | Preguntas eval: 5


## Funciones de retrieval (misma lógica que el Workout 2)


Cada llamada a `recuperar(pregunta, top_k=K)` hace una consulta completa a Chroma y devuelve una **lista de diccionarios** con `text`, `metadata` y `distance`.

In [2]:
def embeddear_consulta(pregunta: str) -> list[float]:
    contents = [types.Content(parts=[types.Part(text=pregunta)])]
    result = client.models.embed_content(model=EMBEDDING_MODEL, contents=contents)
    return list(result.embeddings[0].values)

def recuperar(pregunta: str, top_k: int) -> list[dict]:
    vector = embeddear_consulta(pregunta)
    res = collection.query(
        query_embeddings=[vector],
        n_results=min(top_k, collection.count()),
        include=["documents", "metadatas", "distances"],
    )
    chunks = []
    for i, doc_id in enumerate(res["ids"][0]):
        chunks.append({
            "id": doc_id,
            "text": res["documents"][0][i],
            "metadata": res["metadatas"][0][i],
            "distance": res["distances"][0][i],
        })
    return chunks

def formatear_contexto(chunks: list[dict]) -> str:
    partes = []
    for i, c in enumerate(chunks, 1):
        fuente = Path(c["metadata"].get("source", "?")).name
        partes.append(f"--- Fragmento {i} (dist={c['distance']:.4f}) ---\nFuente: {fuente}\n{c['text']}")
    return "\n\n".join(partes) if partes else "(sin resultados)"

## Experimento 1 — ¿Qué pasa si cambio K?

**K** = número de chunks que devuelve el retriever.

| K bajo (1) | K alto (5) |
|------------|------------|
| Menos ruido, pero puedes perder información | Más cobertura, pero puede entrar texto irrelevante |

Probamos la misma pregunta con K = 1, 3 y 5. Fíjate en:

- ¿Cambia la **fuente** del mejor resultado?
- ¿La **distancia** del #1 mejora o es similar?

No hay un K perfecto universal: depende del corpus y del tipo de pregunta.

In [3]:
pregunta = "¿Hay cine gratuito?"
for k in [1, 3, 5]:
    chunks = recuperar(pregunta, top_k=k)
    mejor = chunks[0] if chunks else {}
    fuente = Path(mejor.get("metadata", {}).get("source", "?")).name
    print(f"K={k} → hits={len(chunks)} | distance={mejor.get('distance')} | fuente={fuente}")

K=1 → hits=1 | distance=0.39596134424209595 | fuente=206974-4-agenda-eventos-culturales-100-csv.csv
K=3 → hits=3 | distance=0.39596134424209595 | fuente=206974-4-agenda-eventos-culturales-100-csv.csv
K=5 → hits=5 | distance=0.39596134424209595 | fuente=206974-4-agenda-eventos-culturales-100-csv.csv


## Experimento 2 — Tabla comparativa con preguntas fijas

Usamos las primeras preguntas de `preguntas_eval.json`. Cada una incluye una **fuente esperada** (orientativa): no es un examen automático, sino una guía para que **tú** juzgues si el retrieval acertó.

Para cada pregunta anotamos:

- `fuente_esperada` — qué archivo debería aportar la respuesta.
- `mejor_fuente` — de dónde vino el chunk #1 realmente.
- `distance` — señal de similitud (compara entre filas, no como verdad absoluta).

Copia esta tabla a tu informe o a la plantilla de teoría (Bloque 3).

In [4]:
filas = []
for item in preguntas[:4]:
    chunks = recuperar(item["texto"], top_k=3)
    mejor = chunks[0] if chunks else {}
    filas.append({
        "id": item["id"],
        "pregunta": item["texto"][:45],
        "fuente_esperada": item.get("fuente_esperada"),
        "mejor_fuente": Path(mejor.get("metadata", {}).get("source", "?")).name,
        "distance": round(mejor.get("distance", 0), 4),
    })

for f in filas:
    print(f)

{'id': 'q1', 'pregunta': '¿Qué significa el campo GRATUITO en el datase', 'fuente_esperada': 'faq_agenda_cultural.md', 'mejor_fuente': '206974-3-agenda-eventos-culturales-100.pdf', 'distance': 0.23}
{'id': 'q2', 'pregunta': '¿Hay cine gratuito en verano?', 'fuente_esperada': '206974-4-agenda-eventos-culturales-100-csv.csv', 'mejor_fuente': '206974-4-agenda-eventos-culturales-100-csv.csv', 'distance': 0.3536}
{'id': 'q3', 'pregunta': '¿Qué actividades hay en el Parque del Retiro?', 'fuente_esperada': '206974-4-agenda-eventos-culturales-100-csv.csv', 'mejor_fuente': '206974-4-agenda-eventos-culturales-100-csv.csv', 'distance': 0.2494}
{'id': 'q4', 'pregunta': '¿Para qué sirve el PDF del dataset?', 'fuente_esperada': 'faq_agenda_cultural.md', 'mejor_fuente': '206974-3-agenda-eventos-culturales-100.pdf', 'distance': 0.3001}


## Experimento 3 — Pregunta fuera del corpus ("trampa")

Un buen hábito de evaluación es probar preguntas que **no están en tus documentos**.

«¿Cuál es la capital de Francia?» no debería estar en la agenda cultural de Madrid. El sistema igual devolverá *algún* chunk (siempre hay vectores «más cercanos»), pero deberían ser **irrelevantes**.

Esto te entrena para no confiar ciegamente en el retrieval: en Sprint 10 añadirás validaciones y prompts que exijan responder solo con el contexto.

In [5]:
trampa = "¿Cuál es la capital de Francia?"
chunks = recuperar(trampa, top_k=3)
print(formatear_contexto(chunks))

--- Fragmento 1 (dist=0.4747) ---
Fuente: 206974-4-agenda-eventos-culturales-100-csv.csv
Evento: 50º aniversario del Distrito de Latina
Descripción: Exposición Conmemorativa. La exposición dedicada al 50º aniversario del Distrito de Latina, ofrece un interesante recorrido visual por la trasformación urbana y social de este emblemático distrito madrileño y de sus barrios, a través de fotografías históricas y planos. Permite descubrir cómo han evolucionado sus calles, edificios y espacios públicos, reflejando también los cambios en la vida cotidiana de sus vecinos.
Actividad: Actividades en el Auditorio Paco de Lucía
Lugar: Auditorio y sala de exposiciones Paco de Lucía (Latina)
Distrito: LATINA
Fecha: 2026-06-08 00:00:00.0
Gratuito: sí

--- Fragmento 2 (dist=0.4779) ---
Fuente: 206974-4-agenda-eventos-culturales-100-csv.csv
Evento: 90 años de la declaración del Parque del Retiro como Bien de Interés Cultural
Actividad: 90 años de la declaración del Parque del Retiro como Bien de Interés

## Experimento 4 — Ajustar chunking (opcional)

Si muchas preguntas fallan, el problema puede no ser K sino **cómo troceaste** el texto en el Workout 1.

Para experimentar:

1. Abre el [Workout 1](../01_Bases_de_datos_vectoriales/01_crear_base_vectorial_chromadb.ipynb).
2. Cambia `CHUNK_SIZE` (p. ej. de 800 a 400).
3. Re-ejecuta **todas** las celdas (regenera embeddings e índice).
4. Vuelve a este notebook y repite los experimentos.

Apunta tus resultados y debate con la clase lo que te haya salido

## Cierre y siguientes pasos

Has practicado **evaluación cualitativa** del retrieval:

- Barrido de **K**.
- Comparación con **preguntas fijas** y fuentes esperadas.
- Prueba de **pregunta trampa** fuera de corpus.

Con un retrieval que recupera buen contexto, en **Sprint 10** conectarás Gemini para generar la respuesta final.